## BERTopic 기반 토픽 자동 도출 (2/2)

**실습 목표:**

1. `representation_model`(KeyBERTInspired)로 토픽 키워드 품질을 개선할 수 있다
2. `min_cluster_size`와 `min_topic_size`의 차이를 이해하고 하이퍼파라미터를 조정할 수 있다
3. 유사 토픽을 통합(merge/reduce)하여 해석 가능한 토픽으로 정제할 수 있다
4. BERTopic 내장 시각화로 토픽 분석 결과를 효과적으로 전달할 수 있다

In [ ]:
# 필요한 패키지를 한번에 설치합니다
# !pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn matplotlib plotly

# uv 권장! uv 환경에서는 위 pip 대신 터미널에서 uv sync 명령어로 설치하세요:
# uv sync

In [ ]:
# 필요한 라이브러리를 import 합니다
import json                          # JSON 파일 로드용
import os                            # 파일 경로 확인용
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer  # 불용어 처리용
from sentence_transformers import SentenceTransformer  # 문장 임베딩 모델
from umap import UMAP                # 차원 축소
from hdbscan import HDBSCAN          # 밀도 기반 군집화
from bertopic import BERTopic        # 토픽 모델링
import warnings
warnings.filterwarnings('ignore')    # 경고 메시지 숨기기

print("라이브러리 로드 완료")

### 1. 데이터 및 모델 준비

**4— BERTopic 파이프라인 한 줄 요약:**

```
문서 => [1. 임베딩] => [2. UMAP 차원축소] => [3. HDBSCAN 군집화] => [4. c-TF-IDF 키워드] => [5. representation_model 키워드 개선] => 토픽 병합
```

지난 시간에는 각 단계를 하나씩 실행해 보고, BERTopic으로 한번에 학습하는 것까지 진행했습니다.

오늘은 여기서 한 걸음 더 나아가서, **KeyBERTInspired로 키워드를 개선** 하고, **파라미터를 조정** 하고, **토픽을 통합** 하고, **시각화** 하는 방법을 배워봅시다!

In [ ]:
# data 폴더에 저장된 JSON 파일에서 데이터를 로드합니다
# 지난 시간과 동일한 4개 카테고리 데이터입니다
with open("data/20newsgroups_4cat.json", "r", encoding="utf-8") as f:
    newsgroups = json.load(f)

docs_raw = newsgroups["data"]                        # 원본 문서 리스트
labels_raw = np.array(newsgroups["target"])           # 실제 카테고리 라벨 (0~3)
target_names = newsgroups["target_names"]             # 카테고리 이름 리스트

print(f"전체 문서 수: {len(docs_raw):,}건")
print(f"카테고리 수: {len(target_names)}개")
print()

# 카테고리별 문서 수를 DataFrame으로 확인합니다
df_cat = pd.DataFrame({"category": [target_names[l] for l in labels_raw]})

print("카테고리 목록:")
display(df_cat["category"].value_counts().sort_index().to_frame("문서 수"))

In [ ]:
# 로드된 데이터를 DataFrame으로 만들어 실제 내용을 확인합니다
df_raw = pd.DataFrame({
    "category": [target_names[l] for l in labels_raw],
    "text_preview": [doc[:100].replace('\n', ' ') for doc in docs_raw]
})

print("데이터 미리보기 (상위 10건):")
display(df_raw.head(10))

In [ ]:
# 빈 문서와 너무 짧은 문서를 필터링합니다
# — 3회차 노트북과 동일한 기준(최소 30자)으로 필터링합니다
min_length = 30  # 최소 30자 이상

# DataFrame으로 변환하여 필터링합니다
df = pd.DataFrame({
    "text": [doc.strip() for doc in docs_raw],  # 앞뒤 공백 제거
    "label": labels_raw,
    "category": [target_names[l] for l in labels_raw]
})
df["text_length"] = df["text"].str.len()

# 최소 길이 이상인 문서만 남깁니다
df = df[df["text_length"] >= min_length].reset_index(drop=True)

# 이후 단계에서 사용할 변수로 추출합니다
docs = df["text"].tolist()
labels = df["label"].tolist()

# 필터링 결과를 출력합니다
print(f"필터링 전: {len(docs_raw):,}건")
print(f"필터링 후: {len(docs):,}건 (최소 {min_length}자 이상)")
print(f"제거된 문서: {len(docs_raw) - len(docs):,}건")
print()
print("필터링 후 카테고리별 분포:")
display(df["category"].value_counts().sort_index().to_frame("문서 수"))

In [ ]:
# 필터링 결과를 확인합니다
# 위 셀에서 만든 df를 그대로 활용합니다 (category, text_length 컬럼 이미 존재)
print("필터링된 데이터 미리보기 (상위 5건):")
display(df[["category", "text_length", "text"]].head())

print()
print("텍스트 길이 통계:")
display(df["text_length"].describe().to_frame().T)

In [ ]:
# 3회차에서 저장해둔 임베딩을 불러옵니다
# — 임베딩 계산이 전체 과정에서 가장 시간이 오래 걸리는 부분이므로,
#   한 번 계산해서 저장해두고 재사용하는 것이 효율적입니다!
# — 3회차에서 avsolatorio/GIST-small-Embedding-v0 모델로 생성한 384차원 임베딩입니다

embedding_model = SentenceTransformer('avsolatorio/GIST-small-Embedding-v0')

embedding_path = "data/all_embeddings.npy"

if os.path.exists(embedding_path):
    # 저장된 임베딩 파일을 로드합니다 (시간 절약!)
    embeddings = np.load(embedding_path)
    print(f"저장된 임베딩 로드 완료: {embedding_path}")
else:
    # 저장된 파일이 없으면 새로 생성합니다
    print(f"저장된 임베딩이 없어 새로 생성합니다...")
    print(f"전체 {len(docs):,}건 문서 임베딩 생성 중... (약 1~2분 소요)")
    embeddings = embedding_model.encode(docs, show_progress_bar=True)

    # 다음에 바로 로드할 수 있도록 저장합니다
    np.save(embedding_path, embeddings)
    print(f"임베딩 저장 완료: {embedding_path}")

print()
print(f"임베딩 행렬 크기: {embeddings.shape}")
print(f"  - 문서 수: {embeddings.shape[0]:,}")
print(f"  - 벡터 차원: {embeddings.shape[1]}")

# 문서 수와 임베딩 수가 일치하는지 확인합니다
assert len(docs) == embeddings.shape[0], (
    f"문서 수({len(docs)})와 임베딩 수({embeddings.shape[0]})가 일치하지 않습니다. "
    f"3회차와 동일한 전처리 기준으로 필터링했는지 확인하십시오."
)
print(f"  - 문서 수와 임베딩 수 일치 확인 완료")

In [ ]:
# 기본 BERTopic 모델을 먼저 학습합니다
# — 이 결과가 "비교 기준(baseline)"이 됩니다. 이후 파라미터를 바꿔가며 개선해봅시다!

# 불용어 제거 + 희귀/과빈출 단어 필터링을 위한 CountVectorizer 설정
vectorizer = CountVectorizer(
    ngram_range=(1,2),     # 1~2단어 조합을 키워드 후보로 사용
    stop_words="english",  # 영어 불용어 제거
    min_df=2,              # 최소 2개 문서에 등장해야 키워드 후보에 포함
    max_df=0.95,           # 전체 문서의 95% 이상에 등장하는 단어 제거
)

baseline_model = BERTopic(
    embedding_model=embedding_model,  # 임베딩 모델 지정
    vectorizer_model=vectorizer,      # 불용어 + 희귀/과빈출 단어 필터링
    language='english',               # 영어 데이터셋이므로
    verbose=True
)

# embeddings 인자로 사전 계산된 임베딩을 전달하면, 내부에서 다시 계산하지 않습니다
baseline_topics, baseline_probs = baseline_model.fit_transform(docs, embeddings=embeddings)

n_baseline = len(set(baseline_topics)) - (1 if -1 in baseline_topics else 0)
n_baseline_noise = baseline_topics.count(-1)

print()
print(f"기본 모델 학습 완료")
print(f"발견된 토픽 수: {n_baseline}개")
print(f"노이즈 문서 수 (토픽 = -1): {n_baseline_noise:,}건 ({n_baseline_noise/len(docs)*100:.1f}%)")

In [ ]:
# 각 문서에 어떤 토픽이 배정되었는지 샘플로 확인합니다
# df에 이미 category가 있으므로 그대로 활용합니다
df_topic_sample = df[["category", "text"]].head(10).copy()
df_topic_sample.insert(1, "assigned_topic", baseline_topics[:10])
df_topic_sample["text"] = df_topic_sample["text"].str[:60]

print("문서별 토픽 배정 결과 (상위 10건):")
display(df_topic_sample)

In [ ]:
# 초기 토픽 결과를 확인해볼까요?
# — 토픽 -1은 어떤 토픽에도 속하지 않는 "노이즈" 문서입니다

baseline_info = baseline_model.get_topic_info()
print("기본 모델 토픽 요약 테이블:")
print("=" * 70)
display(baseline_info.head(15))

In [ ]:
# 기본 모델 - 주요 토픽별 대표 키워드 (상위 5개)
topic_ids = [t for t in baseline_info['Topic'] if t != -1]

rows = []
for tid in topic_ids[:10]:
    keywords = baseline_model.get_topic(tid)
    count = baseline_info[baseline_info['Topic'] == tid]['Count'].values[0]
    row = {"토픽": tid, "문서 수": count}
    for j, (word, score) in enumerate(keywords[:5]):
        row[f"키워드 {j+1}"] = f"{word} ({score:.4f})"
    rows.append(row)

df_keywords = pd.DataFrame(rows).set_index("토픽")

print("기본 모델 - 주요 토픽별 대표 키워드 (상위 5개):")
display(df_keywords)

### 1.5 토픽 키워드 개선: representation_model

기본 c-TF-IDF 키워드는 **단어 나열** 이라 토픽의 의미를 한눈에 파악하기 어렵습니다.
`representation_model`을 사용하면 c-TF-IDF 후보 키워드를 **재정렬/재생성** 하여 품질을 개선할 수 있습니다.

| 방식 | 원리 | 예시 출력 | 추가 비용 |
|------|------|----------|----------|
| 기본 (c-TF-IDF) | 단어 빈도 기반 점수 계산 | space, nasa, shuttle, orbit | 없음 |
| **KeyBERTInspired** | 후보 키워드를 임베딩하여 토픽과 코사인 유사도 순으로 재정렬 | space exploration, nasa mission | 거의 없음 |
| LLM 기반 (GPT 등) | 키워드 + 대표 문서를 LLM에 전달하여 토픽 이름을 자연어로 생성 | "우주 탐사 및 NASA 프로그램" | API 호출 비용 발생 |

---

**KeyBERTInspired — 작동 원리:**

```
c-TF-IDF 후보 키워드들 => 각 키워드를 임베딩 => 토픽 임베딩과 코사인 유사도 계산 => 상위 N개 선택
```

- 딥러닝 모델이 아니라, 이미 로드된 임베딩 모델을 **재사용** 하여 유사도만 계산하는 알고리즘
- 추가 훈련 없음, 추가 비용 없음
- 실무에서 **가장 많이 사용** 되는 representation_model

**LLM 기반 — 개요:**

- BERTopic은 `OpenAI`, `LangChain` 등을 `representation_model`로 지원합니다
- c-TF-IDF 키워드 + 대표 문서를 LLM에 전달하면, **"우주 탐사 및 NASA 프로그램"** 같은 자연어 토픽 이름을 자동 생성합니다
- API 호출 비용이 발생하고 토픽 수만큼 요청이 필요하므로, 비용 대비 효과를 고려해야 합니다
- 이번 실습에서는 다루지 않지만, 실무에서 **결과 보고용 토픽 라벨링** 에 점점 많이 사용되고 있습니다

| 방식 | 실무 사용도 | 적합한 상황 |
|------|-----------|------------|
| **KeyBERTInspired** | 가장 많이 사용 | 기본 적용 (추가 비용 없이 품질 개선) |
| LLM 기반 | 증가 추세 | 이해하기 좋은 보고서 생성, 토픽 라벨 자동 생성 |

In [ ]:
# representation_model을 적용하여 키워드 품질을 비교합니다
# 이미 학습된 baseline_model의 토픽 구조는 유지하고, 키워드 표현만 교체합니다
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

# 기본 모델의 키워드를 먼저 확인합니다
print("[기본 c-TF-IDF] 토픽 1 키워드:")
print(", ".join([word for word, _ in baseline_model.get_topic(1)]))
print()

# KeyBERTInspired: 토픽 임베딩과 가장 유사한 키워드를 선택합니다
keybert_model = KeyBERTInspired()

# update_topics()로 모델을 다시 학습하지 않고 키워드만 교체합니다
# vectorizer_model은 이미 baseline_model에 설정되어 있으므로 생략합니다
baseline_model.update_topics(docs, representation_model=keybert_model)

print("[KeyBERTInspired] 토픽 1 키워드:")
print(", ".join([word for word, _ in baseline_model.get_topic(1)]))
print()

In [ ]:
# 최종 적용된 KeyBERTInspired 키워드를 토픽별로 확인합니다
topic_ids = [t for t in baseline_model.get_topic_info()["Topic"] if t != -1]

rows = []
for tid in topic_ids[:5]:
    keywords = baseline_model.get_topic(tid)
    count = baseline_model.get_topic_info().query("Topic == @tid")["Count"].values[0]
    row = {"토픽": tid, "문서 수": count}
    for j, (word, score) in enumerate(keywords[:5]):
        row[f"키워드 {j+1}"] = f"{word} ({score:.4f})"
    rows.append(row)

df_repr = pd.DataFrame(rows).set_index("토픽")

pd.set_option("display.max_colwidth", None)
print("KeyBERTInspired 적용 후 토픽별 키워드 (상위 5개 토픽):")
display(df_repr)
pd.reset_option("display.max_colwidth")

print()
print("=> update_topics()를 사용하면 모델을 다시 학습하지 않고 키워드만 교체할 수 있습니다.")
print("=> 실무에서는 KeyBERTInspired가 가장 널리 사용됩니다.")

### 2. 하이퍼파라미터 조정

BERTopic의 결과는 내부 파라미터에 따라 크게 달라질 수 있습니다.

#### 핵심 파라미터 정리

| 파라미터 | 위치 | 효과 |
|----------|------|------|
| `n_neighbors` | UMAP | 이웃 수. 클수록 전역 구조 보존, 작을수록 지역 구조 보존 |
| `n_components` | UMAP | 차원 축소 목표 차원. 보통 5~10 |
| `min_cluster_size` | HDBSCAN | 군집 최소 크기. 이 수 미만이면 **노이즈(-1)로 버림** |
| `min_samples` | HDBSCAN | 핵심 포인트 판별 기준. 클수록 => 노이즈 증가 |
| `min_topic_size` | BERTopic | HDBSCAN 이후, 이 수 미만인 토픽을 **가장 가까운 토픽에 병합** |
| `nr_topics` | BERTopic | 최종 토픽 수를 직접 지정. 자동 결정 후 병합하여 맞춤 |

---

#### `min_cluster_size` vs `min_topic_size` — 헷갈리기 쉬운 핵심 차이

이름이 비슷해서 혼동하기 쉽지만, **작동 시점과 처리 방식이 완전히 다릅니다.**

| | `min_cluster_size` (HDBSCAN) | `min_topic_size` (BERTopic) |
|---|---|---|
| **작동 시점** | 1단계: HDBSCAN 군집화 수행 시 | 2단계: 군집화 **이후** 토픽 정리 시 |
| **판단 기준** | "이건 군집이 아니다" | "이 토픽은 너무 작다" |
| **처리 방식** | 기준 미달 문서를 **노이즈(-1)로 버림** | 기준 미달 토픽을 **가장 가까운 토픽에 병합** |
| **문서 손실** | 있음 (노이즈 처리) | 없음 (다른 토픽에 흡수) |

#### (1) min_topic_size 비교: 10 vs 30 vs 50

`min_topic_size`는 **"토픽 하나에 최소 몇 개의 문서가 있어야 하는가"** 를 결정합니다.

- **작은 값 (예: 10)**: 세부적인 토픽이 많이 나옴 => 해석이 어려울 수 있음
- **큰 값 (예: 50)**: 큰 토픽만 남음 => 세부 정보가 사라질 수 있음

세 가지 값으로 비교해봅시다!

In [ ]:
# min_topic_size를 10, 30, 50으로 바꿔가며 BERTopic을 학습합니다
# 임베딩은 이미 계산해둔 것을 재사용하므로 빠르게 실행됩니다

min_sizes = [10, 30, 50]
results = {}  # 결과를 저장할 딕셔너리

for size in min_sizes:
    print(f"\n{'='*50}")
    print(f"min_topic_size = {size} 로 학습 중...")
    print(f"{'='*50}")
    
    # BERTopic 모델 생성 — min_topic_size만 변경합니다
    model = BERTopic(
        embedding_model=embedding_model,
        min_topic_size=size,
        vectorizer_model=CountVectorizer(stop_words="english", min_df=2, max_df=0.95),
        representation_model=KeyBERTInspired(),  # 키워드 품질 개선
        language='english',
        verbose=False
    )
    
    # 사전 계산된 임베딩을 전달하여 학습
    topics, probs = model.fit_transform(docs, embeddings=embeddings)
    
    # 결과 저장
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_noise = topics.count(-1)
    noise_pct = n_noise / len(topics) * 100
    
    results[size] = {
        'model': model,
        'topics': topics,
        'n_topics': n_topics,
        'n_noise': n_noise,
        'noise_pct': noise_pct
    }
    
    print(f"  토픽 수: {n_topics}개")
    print(f"  노이즈 문서: {n_noise}건 ({noise_pct:.1f}%)")

#### (2) UMAP / HDBSCAN 커스텀 객체 전달

BERTopic에 **직접 만든 UMAP, HDBSCAN 객체** 를 전달할 수 있습니다.
이렇게 하면 각 단계의 파라미터를 더 세밀하게 제어할 수 있습니다!

- `UMAP(n_neighbors=30)`: 전체 구조를 더 잘 보존 (큰 군집 발견에 유리)
- `HDBSCAN(min_cluster_size=30)`: 중간 크기 이상의 군집만 남김

In [ ]:
# UMAP과 HDBSCAN 객체를 직접 만들어서 BERTopic에 전달합니다

# 커스텀 UMAP — n_neighbors를 30으로 늘려서 전역 구조를 강조합니다
custom_umap = UMAP(
    n_components=5,       # 5차원으로 축소 (군집화에 적합)
    n_neighbors=30,       # 이웃 수를 늘려서 전역 구조 반영
    min_dist=0.0,         # 점들이 가깝게 모이도록 허용
    metric='cosine',      # 텍스트 임베딩에 적합한 코사인 거리 사용
    random_state=42
)

# 커스텀 HDBSCAN — min_cluster_size를 30으로 설정
# metric: 점 간 거리를 계산할 때 사용하는 거리 함수
#   - 'euclidean' (기본값): 유클리드 거리. UMAP 축소 후 저차원 공간에서 주로 사용
#   - 'manhattan': 맨해튼 거리. 격자형 데이터에 적합
#   - 'cosine': 코사인 거리. 고차원 임베딩에 직접 HDBSCAN을 적용할 때 유용
#   => UMAP으로 차원 축소 후에는 'euclidean'이 표준 선택입니다
custom_hdbscan = HDBSCAN(
    min_cluster_size=30,  # 최소 30개 문서가 있어야 군집으로 인정
    min_samples=10,       # 핵심 포인트 판별 기준을 좀 더 엄격하게
    metric='euclidean',   # 유클리드 거리 (UMAP 축소 후 표준 선택)
    prediction_data=True  # 나중에 새 문서 예측 시 필요
)

print("커스텀 UMAP, HDBSCAN 객체 생성 완료")

In [ ]:
# 커스텀 모델로 BERTopic 학습
custom_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=custom_umap,       # 커스텀 UMAP 전달
    hdbscan_model=custom_hdbscan, # 커스텀 HDBSCAN 전달
    vectorizer_model=CountVectorizer(stop_words="english", min_df=2, max_df=0.95),
    representation_model=KeyBERTInspired(),  # 키워드 품질 개선
    language='english',
    verbose=True
)

print("커스텀 모델 학습 중...")
custom_topics, custom_probs = custom_model.fit_transform(docs, embeddings=embeddings)

n_custom = len(set(custom_topics)) - (1 if -1 in custom_topics else 0)
n_custom_noise = custom_topics.count(-1)

print()
print(f"커스텀 모델 학습 완료")
print(f"발견된 토픽 수: {n_custom}개")
print(f"노이즈 문서 수: {n_custom_noise}건 ({n_custom_noise / len(custom_topics) * 100:.1f}%)")

In [ ]:
# 기본 모델 vs 커스텀 모델을 비교해봅시다

print("기본 모델 vs 커스텀 모델 비교:")
print("=" * 55)
print(f"{'항목':<20} {'기본 모델':<15} {'커스텀 모델'}")
print("-" * 55)
print(f"{'토픽 수':<20} {n_baseline:<15} {n_custom}")
print(f"{'노이즈 수':<20} {n_baseline_noise:<15} {n_custom_noise}")
print(f"{'노이즈 비율':<20} {n_baseline_noise/len(docs)*100:<14.1f}% {n_custom_noise/len(docs)*100:.1f}%")

#### (3) nr_topics로 토픽 수 직접 지정

때로는 **"토픽이 정확히 몇 개 필요하다"** 는 요구사항이 있을 수 있습니다.

`nr_topics` 파라미터를 사용하면, BERTopic이 먼저 자동으로 토픽을 찾은 뒤,
유사한 토픽끼리 **자동 병합** 하여 지정한 수로 줄여줍니다.

> **주의**: `nr_topics`는 **노이즈(-1)를 포함한** 전체 그룹 수입니다!
> - `nr_topics=5` => 노이즈(-1) + 실제 토픽 4개
> - 실제 토픽 5개를 원하면 `nr_topics=6`으로 설정해야 합니다

우리 데이터는 4개 카테고리이므로, `nr_topics=5`로 설정해봅시다. (노이즈 1 + 실제 토픽 4개)

In [ ]:
# nr_topics=5로 토픽 수를 직접 지정합니다
# — BERTopic이 먼저 토픽을 자동으로 찾은 뒤, 5개로 자동 병합합니다
# — 주의: nr_topics는 노이즈(-1)를 포함한 수이므로, 실제 토픽은 4개가 됩니다

nr_model = BERTopic(
    embedding_model=embedding_model,
    nr_topics=5,           # 노이즈(-1) 포함 5개 => 실제 토픽 4개
    vectorizer_model=CountVectorizer(stop_words="english", min_df=2, max_df=0.95),
    representation_model=KeyBERTInspired(),  # 키워드 품질 개선
    language='english',
    verbose=True
)

print("nr_topics=5 모델 학습 중...")
nr_topics, nr_probs = nr_model.fit_transform(docs, embeddings=embeddings)

n_nr = len(set(nr_topics)) - (1 if -1 in nr_topics else 0)
print()
print(f"학습 완료 — 최종 토픽 수: {n_nr}개 (nr_topics=5에서 노이즈 제외)")

In [ ]:
# nr_topics=5 모델의 토픽 키워드를 확인하겠습니다
nr_info = nr_model.get_topic_info()

print("nr_topics=5 모델 - 전체 토픽 요약:")
display(nr_info)
print()

# 각 토픽의 대표 키워드를 DataFrame으로 정리합니다
topic_ids = [t for t in nr_info['Topic'] if t != -1]

rows = []
for tid in topic_ids:
    keywords = nr_model.get_topic(tid)
    count = nr_info[nr_info['Topic'] == tid]['Count'].values[0]
    row = {"토픽": tid, "문서 수": count}
    for j, (word, score) in enumerate(keywords[:7]):
        row[f"키워드 {j+1}"] = f"{word} ({score:.4f})"
    rows.append(row)

df_keywords = pd.DataFrame(rows).set_index("토픽")

print("각 토픽의 대표 키워드 (상위 7개):")
display(df_keywords)


### 3. 토픽 통합 (Merge)

BERTopic이 자동으로 찾은 토픽 중에는 **서로 비슷한 토픽** 이 있을 수 있습니다.

예를 들어 살펴볼까요?
- 토픽 1: "shuttle, launch, nasa" (우주 발사)
- 토픽 3: "orbit, moon, mission" (우주 탐사)

이 두 토픽은 모두 "우주 과학"에 해당하므로 하나로 합치는 것이 더 깔끔합니다!

**통합 방법 2가지:**

| 방법 | 함수 | 설명 |
|------|------|------|
| 수동 통합 | `merge_topics()` | 합칠 토픽 번호를 직접 지정 |
| 자동 축소 | `reduce_topics()` | 원하는 토픽 수를 지정하면 유사한 토픽끼리 자동 병합 |

![토픽 통합 개념](images/4_topic_merge.png)

#### (1) 수동 통합: merge_topics()

키워드를 보고 **"이 토픽들은 같은 주제다"** 라고 판단되면, 직접 합칠 수 있습니다.

> **주의**: `merge_topics()`는 모델을 **in-place로 수정** 합니다 (원본이 바뀜).
> 그래서 별도 모델을 만들어서 실험하는 것이 안전합니다!

In [ ]:
# 수동 통합 실습을 위해, min_topic_size=10으로 토픽을 많이 만들겠습니다
# — 토픽이 많아야 통합 연습을 해볼 수 있기 때문입니다

merge_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=10,  # 작은 값 => 토픽이 많이 생성됨
    vectorizer_model=CountVectorizer(stop_words="english", min_df=2, max_df=0.95),
    representation_model=KeyBERTInspired(),  # 키워드 품질 개선
    language='english',
    verbose=False
)

merge_topics_result, merge_probs = merge_model.fit_transform(docs, embeddings=embeddings)

# 통합 전 토픽 정보 확인
merge_info_before = merge_model.get_topic_info()
valid_topics_before = [t for t in merge_info_before['Topic'] if t != -1]
n_before = len(valid_topics_before)

print(f"통합 전 토픽 수: {n_before}개")
print()

# 토픽 키워드를 DataFrame으로 정리합니다
rows = []
for _, row in merge_info_before.iterrows():
    topic_id = row['Topic']
    if topic_id == -1:
        continue
    keywords = merge_model.get_topic(topic_id)
    keyword_str = ", ".join([word for word, score in keywords[:5]])
    rows.append({"topic": topic_id, "count": row['Count'], "keywords": keyword_str})

df_merge_keywords = pd.DataFrame(rows)

print("토픽 키워드 (통합 전):")
display(df_merge_keywords)

In [ ]:
# 위의 키워드를 보고, 비슷한 토픽을 묶어봅시다!
# 실행 결과에 따라 토픽 번호가 달라질 수 있으니, 위 출력을 참고해서 조정하십시오

# 현재 유효한 토픽 ID 목록 확인
valid_ids = sorted([t for t in set(merge_topics_result) if t != -1])
print(f"현재 토픽 ID: {valid_ids}")

# 예시: 처음 두 토픽과 그 다음 두 토픽을 각각 통합
# (실제로는 키워드를 보고 의미적으로 비슷한 토픽끼리 묶어야 합니다!)

topics_to_merge = [
    [0, 1],  # 비슷한 주제의 토픽 묶기
    [10, 13],  # 비슷한 주제의 토픽 묶기
]

print(f"통합할 토픽 그룹: {topics_to_merge}")
print()
print("merge_topics() 실행 중...")

# merge_topics()로 수동 통합 수행
merge_model.merge_topics(docs, topics_to_merge)

print("통합 완료!")


In [ ]:
# 통합 전후 결과를 비교합니다
merge_info_after = merge_model.get_topic_info()
n_after = len([t for t in merge_info_after['Topic'] if t != -1])

print(f"통합 전: {n_before}개 => 통합 후: {n_after}개")
print()

# 통합 후 토픽 키워드를 DataFrame으로 정리합니다
rows = []
for _, row in merge_info_after.iterrows():
    topic_id = row['Topic']
    if topic_id == -1:
        continue
    keywords = merge_model.get_topic(topic_id)
    keyword_str = ", ".join([word for word, score in keywords[:5]])
    rows.append({"topic": topic_id, "count": row['Count'], "keywords": keyword_str})

df_merge_after = pd.DataFrame(rows)

print("토픽 키워드 (통합 후):")
display(df_merge_after)

#### (2) 자동 축소: reduce_topics()

일일이 토픽 번호를 확인하고 묶는 것이 번거롭다면,
`reduce_topics()`를 사용하면 원하는 개수로 **자동 병합** 해줍니다!

내부적으로 c-TF-IDF 벡터 간 유사도를 계산해서, 가장 비슷한 토픽부터 순차적으로 합칩니다.

In [ ]:
# reduce_topics()로 토픽 수를 5개로 자동 축소하겠습니다
# 새 모델을 만들어서 실험합니다 (기존 모델은 보존!)

reduce_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=10,  # 처음에 토픽을 많이 만든 뒤 축소하는 전략입니다
    vectorizer_model=CountVectorizer(stop_words="english", min_df=2, max_df=0.95),
    representation_model=KeyBERTInspired(),  # 키워드 품질 개선
    language='english',
    verbose=False
)

reduce_topics_result, reduce_probs = reduce_model.fit_transform(docs, embeddings=embeddings)

n_before_reduce = len(set(reduce_topics_result)) - (1 if -1 in reduce_topics_result else 0)
print(f"축소 전 토픽 수: {n_before_reduce}개")

print()
print(f"reduce_topics(nr_topics=5) 실행 중...")
reduce_model.reduce_topics(docs, nr_topics=5)

# 축소 후 결과 확인
reduce_info = reduce_model.get_topic_info()
n_after_reduce = len([t for t in reduce_info['Topic'] if t != -1])

print()
print(f"축소 완료!")
print(f"축소 전: {n_before_reduce}개 => 축소 후: {n_after_reduce}개")

In [ ]:
# 축소 후 토픽 키워드를 DataFrame으로 확인합니다
topic_ids = [t for t in reduce_info['Topic'] if t != -1]

rows = []
for tid in topic_ids:
    keywords = reduce_model.get_topic(tid)
    count = reduce_info[reduce_info['Topic'] == tid]['Count'].values[0]
    row = {"토픽": tid, "문서 수": count}
    for j, (word, score) in enumerate(keywords[:7]):
        row[f"키워드 {j+1}"] = f"{word} ({score:.4f})"
    rows.append(row)

df_keywords = pd.DataFrame(rows).set_index("토픽")

print("축소 후 토픽 키워드:")
display(df_keywords)

### 4. BERTopic 시각화

BERTopic은 **plotly 기반의 인터랙티브 시각화** 를 내장하고 있습니다.
마우스를 올려서 세부 정보를 확인할 수 있는 차트가 자동으로 만들어집니다!

이번 섹션에서는 5가지 핵심 시각화를 살펴봅시다.

| 시각화 | 함수 | 확인할 수 있는 것 |
|--------|------|------------------|
| 토픽 간 거리 맵 | `visualize_topics()` | 토픽 간의 거리와 크기 |
| 키워드 막대 차트 | `visualize_barchart()` | 각 토픽의 대표 키워드와 중요도 |
| 토픽 유사도 히트맵 | `visualize_heatmap()` | 토픽 간 유사도를 색상으로 표현 |
| 토픽 계층 구조 | `visualize_hierarchy()` | 토픽 간의 계층적 관계 |
| 문서-토픽 분포 | `visualize_documents()` | 개별 문서가 어떤 토픽에 속하는지 |

> **참고**: 시각화에는 `reduce_topics()`로 축소한 모델을 사용하겠습니다. 토픽 수가 적당해야 보기 편합니다!

#### (1) 토픽 간 거리 맵: visualize_topics()

토픽들을 **2D 공간에 원(circle)으로 표시** 합니다.
- 원의 크기: 해당 토픽의 문서 수 (클수록 문서가 많음)
- 원 사이의 거리: 토픽 간 유사도 (가까울수록 비슷한 주제)
- 마우스를 올리면 토픽의 대표 키워드를 확인할 수 있습니다!

In [ ]:
# 토픽 간 거리 맵 — 토픽들이 2D 공간에서 어디에 위치하는지 확인합니다
# 가까이 있는 토픽은 서로 비슷한 주제라는 뜻입니다
# 마우스를 올려보면 각 토픽의 키워드를 확인할 수 있습니다

fig_topics = reduce_model.visualize_topics()
fig_topics.show()

#### (2) 토픽별 키워드 막대 차트: visualize_barchart()

각 토픽의 **대표 키워드와 c-TF-IDF 점수** 를 막대 그래프로 보여줍니다.
- 막대가 길수록 해당 키워드가 토픽을 잘 대표하는 것입니다
- 보고서나 발표 자료에 넣기 가장 좋은 시각화입니다!

In [ ]:
# 토픽별 키워드 막대 차트 — 각 토픽의 핵심 키워드를 확인합니다
# top_n_topics: 상위 몇 개 토픽을 표시할지
# n_words: 토픽당 몇 개 키워드를 보여줄지

fig_barchart = reduce_model.visualize_barchart(
    top_n_topics=8,  # 상위 8개 토픽 표시
    n_words=8         # 토픽당 8개 키워드 표시
)
fig_barchart.show()

#### (3) 토픽 유사도 히트맵: visualize_heatmap()

토픽 간의 **유사도를 색상 행렬** 로 보여줍니다.
- 진한 색: 유사도가 높음 (비슷한 토픽)
- 연한 색: 유사도가 낮음 (다른 토픽)
- 토픽 통합(merge) 시 어떤 토픽을 합칠지 판단하는 데 유용합니다!

In [ ]:
# 토픽 유사도 히트맵 — 어떤 토픽끼리 비슷한지 색상으로 확인합니다
# 진한 색일수록 유사도가 높습니다 => merge할 때 참고하면 좋습니다!

fig_heatmap = reduce_model.visualize_heatmap()
fig_heatmap.show()

#### (4) 토픽 계층 구조: visualize_hierarchy()

토픽들의 **계층적 관계(덴드로그램)** 를 보여줍니다.
- 아래쪽에서 일찍 합쳐지는 토픽 = 서로 매우 유사
- 위쪽에서 늦게 합쳐지는 토픽 = 서로 다른 주제
- 토픽을 몇 개로 줄일지 결정할 때 이 그래프를 참고하면 좋습니다!

In [ ]:
# 토픽 계층 구조 — 토픽들이 어떻게 묶이는지 트리 형태로 확인합니다
# 가까이에서 합쳐지는 토픽들은 서로 비슷한 주제입니다

fig_hierarchy = reduce_model.visualize_hierarchy()
fig_hierarchy.show()

#### (5) 문서-토픽 분포: visualize_documents()

**개별 문서를 2D 공간에 점으로 표시** 하고, 각 점의 색상으로 토픽을 나타냅니다.
- 같은 색 점이 모여 있으면 => 해당 토픽이 잘 분리된 것
- 여러 색이 섞여 있으면 => 토픽 경계가 모호한 영역
- 마우스를 올리면 문서 내용 미리보기와 토픽 정보를 확인할 수 있습니다!

In [ ]:
# 문서-토픽 분포 시각화 — 각 문서가 어떤 토픽에 배정되었는지 2D로 확인합니다
# embeddings를 전달하면 UMAP으로 자동 차원 축소를 해줍니다

# 문서가 많으면 마우스 오버 시 텍스트가 너무 길어질 수 있으므로,
# 100자까지만 잘라서 전달합니다
docs_short = [doc[:100] for doc in docs]

fig_docs = reduce_model.visualize_documents(
    docs_short,
    embeddings=embeddings  # 사전 계산된 임베딩 전달
)
fig_docs.show()

### 5. 정리 및 핵심 요약

**이번 실습에서 배운 내용을 정리하겠습니다!**

---

**1. 하이퍼파라미터 조정 가이드**

| 이런 상황이라면 | 이렇게 조정해보십시오 |
|-----------------|--------------------|
| 토픽이 너무 많다 | `min_topic_size` 증가 (10 => 30~50) |
| 토픽이 너무 적다 | `min_topic_size` 감소, HDBSCAN `min_cluster_size` 감소 |
| 노이즈(이상치)가 너무 많다 | HDBSCAN `min_samples` 감소 |
| 토픽 수를 정확히 맞추고 싶다 | `nr_topics`로 목표 수 지정 또는 `reduce_topics()` 사용 |
| 전역 구조를 더 잘 반영하고 싶다 | UMAP `n_neighbors` 증가 (15 => 30~50) |
| 지역 구조(세부 토픽)를 보고 싶다 | UMAP `n_neighbors` 감소 |

**2. 토픽 통합(Merge) 전략**

| 방법 | 함수 | 언제 사용할까요? |
|------|------|-------------|
| 수동 통합 | `merge_topics(docs, [[id1, id2]])` | 키워드를 보고 직접 판단할 때 |
| 자동 축소 | `reduce_topics(docs, nr_topics=N)` | 원하는 토픽 수만 정해두고 싶을 때 |

**3. 시각화별 활용 장면**

| 시각화 | 언제 사용할까요? |
|--------|-------------|
| `visualize_topics()` | 토픽 전체 구조를 파악할 때 |
| `visualize_barchart()` | 토픽의 의미를 키워드로 해석할 때 (발표/보고서에 최적!) |
| `visualize_heatmap()` | 어떤 토픽을 합칠지 판단할 때 |
| `visualize_hierarchy()` | 토픽을 몇 개로 줄일지 결정할 때 |
| `visualize_documents()` | 문서 수준에서 토픽 분리 품질을 확인할 때 |

**4. 실전 팁**
- 임베딩은 **한 번만 계산** 하고 재사용하면 실험 속도가 크게 향상됩니다!
- `min_topic_size`를 먼저 조정하고, 필요하면 UMAP/HDBSCAN을 세밀 조정하십시오
- 시각화를 적극 활용하여 **토픽 품질을 눈으로 확인** 하는 습관을 들여봅시다!

---

**다음 시간 예고:** IMDB 영화 리뷰 데이터로 실전 분석을 진행합니다! 지금까지 배운 텍스트 분석 기법(전처리, 빈도 분석, 감성 분류, 토픽 모델링)을 종합적으로 적용해봅시다!